# ACB - database to spreadsheet
## Ana Carolina Brandão, Abril 2026

Este script serve para exportar automaticamente para Google Spreadsheets as tabelas da base de dados cujo nome contém `catalog`.

O objetivo é permitir consultar e validar os dados existentes na base de dados de forma mais simples, através de uma spreadsheet. Para isso, o script estabelece ligação à base de dados, identifica as tabelas relevantes, lê o seu conteúdo e exporta cada tabela para uma folha diferente da Google Spreadsheet.

As credenciais são carregadas através de secrets do Colab, usando dinamicamente as iniciais do nome do notebook para identificar o utilizador.


In [11]:
## imports
##

import json
import os
import socket
import requests
import re
from datetime import datetime

current_env = os.environ.get('CONDA_DEFAULT_ENV')
print("current_env:", current_env)

if current_env is None:
    !pip install pymysql --quiet
    !pip install clts_pcp --quiet
    !pip install pandas --quiet
    !pip install gspread --quiet
    !pip install gspread-dataframe --quiet
    !pip install google-auth --quiet
    !pip install pytz --quiet

    from google.colab import userdata
    from google.colab import auth
    from google.auth import default

import pymysql
import pandas as pd
import clts_pcp as clts
import gspread
import pytz
from gspread_dataframe import set_with_dataframe

print("... done.")

current_env: None
... done.


In [12]:
## Context gathering
##

tstart = clts.getts()

DEFAULT_PARAMS = {
    "verbose": True,
    "timeout": 20,
    "spreadsheet_name": "database_export",
    "share_email": "ana.carolinamb2001@gmail.com",
    "spreadsheet_fixa_id": "1FTPlF_maNoTUl6kmMkqHexZvVgWR6VCvnRrBw0jJ16o",
    "filtro_tabelas": "catalog" #filtro do nome das tabelas Ex: catalog -> todas as tabelas com catalog no nome
}

verbose = DEFAULT_PARAMS["verbose"]
timeout = DEFAULT_PARAMS["timeout"]
spreadsheet_name = DEFAULT_PARAMS["spreadsheet_name"]
share_email = DEFAULT_PARAMS["share_email"]
spreadsheet_fixa_id = DEFAULT_PARAMS["spreadsheet_fixa_id"]
filtro_tabelas = DEFAULT_PARAMS["filtro_tabelas"]

hostname = socket.gethostname()

try:
    ip = requests.get("https://api.ipify.org", timeout=5).text
except:
    ip = "unknown"

print("Server name:", hostname, "Public IP Address:", ip)

if "__file__" in globals():
    enviro = "airflow/linux"
    script = os.path.basename(__file__)
    parts = __file__.replace("\\", "/").split("/")
    channel = parts[-2] if len(parts) >= 2 else "unknown"
else:
    enviro = "jupyter"
    channel = "colab"
    script = requests.get("http://172.28.0.12:9000/api/sessions").json()[0]["name"]

match = re.match(r"([A-Za-z]{3})", script)

if not match:
    raise ValueError(f"Não foi possível identificar o utilizador a partir do nome do ficheiro: {script}")

user = match.group(1).lower()

context = f"{hostname} ({ip}) | {user} | {channel} | {script}"
clts.setcontext(context)

if verbose:
    print("script:", script)
    print("user:", user)
    print("context:", context)

Server name: 88ad2830168a Public IP Address: 8.231.223.30
script: ACB-database_to_spreadsheet_Teste.ipynb
user: acb
context: 88ad2830168a (8.231.223.30) | acb | colab | ACB-database_to_spreadsheet_Teste.ipynb


In [13]:
## Ler segredos da base de dados
##

db_secret_name = f"{user}-d5hive-aiven-super-1.json"

dbcreds_json = userdata.get(db_secret_name)

if dbcreds_json is None:
    raise ValueError(f"Secret não encontrado no Colab: {db_secret_name}")

dbcreds = json.loads(dbcreds_json)

DB_HOST = dbcreds["dest_host"]
DB_PORT = int(dbcreds["port"])
DB_NAME = dbcreds["database"]
DB_USER = dbcreds["username"]
DB_PASSWORD = dbcreds["password"]

print("Segredos carregados com sucesso.")
print("Secret usado:", db_secret_name)
print("DB_HOST:", DB_HOST)
print("DB_PORT:", DB_PORT)
print("DB_NAME:", DB_NAME)
print("DB_USER:", DB_USER)

Segredos carregados com sucesso.
Secret usado: acb-d5hive-aiven-super-1.json
DB_HOST: d5hive-super-aiven-1-dfivehive-a1dd.c.aivencloud.com
DB_PORT: 19861
DB_NAME: d5hive
DB_USER: anacarolina


In [14]:
## Autenticação Google
##

def autenticar_google():
    try:
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)

        print("Autenticação Google com sucesso!")
        clts.elapt["Google authentication successful ✅"] = clts.deltat(tstart)
        return gc

    except Exception as e:
        print("Erro na autenticação Google:", e)
        clts.elapt[f"Google authentication error ❌: {e}"] = clts.deltat(tstart)
        return None

In [15]:
## Ligação à base de dados
##

def ligar_bd():
    try:
        connection = pymysql.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            cursorclass=pymysql.cursors.DictCursor,
            charset="utf8mb4",
            connect_timeout=timeout,
            read_timeout=timeout,
            write_timeout=timeout,
            autocommit=True
        )

        print("Ligação à base de dados com sucesso!")
        clts.elapt["Database connection successful ✅"] = clts.deltat(tstart)
        return connection

    except Exception as e:
        print("Erro na ligação à base de dados:", e)
        clts.elapt[f"Database connection error ❌: {e}"] = clts.deltat(tstart)
        return None

In [16]:
## Mostrar tabelas
##

def listar_tabelas(conn, filtro_nome="catalog"):
    try:
        cursor = conn.cursor()

        query = """
        SELECT table_name AS nome_tabela
        FROM information_schema.tables
        WHERE table_schema = %s
          AND table_name LIKE %s
        ORDER BY table_name;
        """
        cursor.execute(query, (DB_NAME, f"%{filtro_nome}%"))
        tabelas = cursor.fetchall()

        print(f"\nTabelas existentes na base de dados com '{filtro_nome}' no nome:\n")

        if not tabelas:
            print("Não foram encontradas tabelas com esse filtro.")
            return []

        nomes_tabelas = []

        for i, row in enumerate(tabelas, start=1):
            nome_tabela = row["nome_tabela"]
            nomes_tabelas.append(nome_tabela)

            query_colunas = """
            SELECT COUNT(*) AS nr_colunas
            FROM information_schema.columns
            WHERE table_schema = %s
              AND table_name = %s;
            """
            cursor.execute(query_colunas, (DB_NAME, nome_tabela))
            resultado = cursor.fetchone()
            nr_colunas = resultado["nr_colunas"]

            print(f"{i}. {nome_tabela} - {nr_colunas} colunas")

        clts.elapt["Tables listed successfully ✅"] = clts.deltat(tstart)
        return nomes_tabelas

    except Exception as e:
        print("Erro ao listar tabelas:", e)
        clts.elapt[f"List tables error ❌: {e}"] = clts.deltat(tstart)
        return []

In [17]:
## Preparação e gestão das folhas da spreadsheet
##

def limpar_nome_folha(nome):
    nome = re.sub(r'[:\\/*?\[\]]', '_', str(nome))
    nome = nome.strip()

    if len(nome) > 100:
        nome = nome[:100]

    if nome == "":
        nome = "Sheet"

    return nome

def obter_ou_criar_folha(spreadsheet, nome_folha, n_linhas=100, n_colunas=20):
    try:
        worksheet = spreadsheet.worksheet(nome_folha)
        worksheet.clear()
        print(f"Folha existente limpa: {nome_folha}")
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=nome_folha,
            rows=str(n_linhas),
            cols=str(n_colunas)
        )
        print(f"Folha criada: {nome_folha}")

    return worksheet

In [18]:
## Ler uma tabela da BD para DataFrame
##

def tabela_para_dataframe(conn, nome_tabela):
    try:
        cursor = conn.cursor()
        query = f"SELECT * FROM `{nome_tabela}`;"
        cursor.execute(query)

        rows = cursor.fetchall()

        if rows:
            df = pd.DataFrame(rows)
        else:
            query_colunas = f"SHOW COLUMNS FROM `{nome_tabela}`;"
            cursor.execute(query_colunas)
            colunas_info = cursor.fetchall()
            nomes_colunas = [col["Field"] for col in colunas_info]
            df = pd.DataFrame(columns=nomes_colunas)

        return df

    except Exception as e:
        print(f"Erro ao ler a tabela {nome_tabela} para DataFrame:", e)
        return None

In [19]:
## Exportar tabelas para uma spreadsheet
##

def exportar_para_uma_spreadsheet(conn, spreadsheet, tabelas, apagar_sheet1=False):
    try:
        if not tabelas:
            print(f"Não existem tabelas para exportar na spreadsheet '{spreadsheet.title}'.")
            return pd.DataFrame()

        resumo = []

        for nome_tabela in tabelas:
            print(f"\nA exportar tabela: {nome_tabela} -> {spreadsheet.title}")

            try:
                df = tabela_para_dataframe(conn, nome_tabela)

                if df is None:
                    raise Exception("Falha na leitura da tabela")

                if verbose:
                    print(df.head())
                    print("shape:", df.shape)

                nome_folha = limpar_nome_folha(nome_tabela)

                worksheet = obter_ou_criar_folha(
                    spreadsheet,
                    nome_folha,
                    n_linhas=max(len(df) + 10, 100),
                    n_colunas=max(len(df.columns) + 5, 20)
                )

                set_with_dataframe(
                    worksheet,
                    df,
                    include_index=False,
                    include_column_header=True,
                    resize=True
                )

                resumo.append({
                    "spreadsheet": spreadsheet.title,
                    "tabela": nome_tabela,
                    "linhas": len(df),
                    "colunas": len(df.columns),
                    "estado": "ok"
                })

                print(f"Tabela exportada com sucesso: {nome_tabela} ({len(df)} linhas)")

            except Exception as e:
                print(f"Erro ao exportar tabela {nome_tabela}: {e}")

                resumo.append({
                    "spreadsheet": spreadsheet.title,
                    "tabela": nome_tabela,
                    "linhas": None,
                    "colunas": None,
                    "estado": f"erro: {e}"
                })

        if apagar_sheet1:
            try:
                sheet1 = spreadsheet.sheet1
                if sheet1.title in ["Sheet1", "Folha1"]:
                    spreadsheet.del_worksheet(sheet1)
                    print(f"Folha inicial apagada: {sheet1.title}")
            except Exception as e:
                print(f"Não foi possível apagar a folha inicial: {e}")

        df_resumo = pd.DataFrame(resumo)

        print("\nResumo da exportação:\n")
        print(df_resumo)
        print("URL da spreadsheet:", spreadsheet.url)

        return df_resumo

    except Exception as e:
        print(f"Erro geral na exportação para a spreadsheet '{spreadsheet.title}':", e)
        return pd.DataFrame()

In [20]:
## Execução
##

conn = None

try:
    # 1) Ligar à base de dados
    conn = ligar_bd()

    if conn is None:
        raise Exception("Não foi possível estabelecer ligação à base de dados.")

    # 2) Autenticar no Google
    gc = autenticar_google()

    if gc is None:
        raise Exception("Não foi possível autenticar no Google.")

    # 3) Listar tabelas da base de dados
    tabelas = listar_tabelas(conn, filtro_nome=filtro_tabelas)

    if not tabelas:
        raise Exception(f"Não foram encontradas tabelas com '{filtro_tabelas}' no nome.")

    # 4) Criar spreadsheet nova
    tz = pytz.timezone("Europe/Lisbon")
    timestamp = datetime.now(tz).strftime("%Y%m%d_%H%M%S")
    nome_spreadsheet = f"{spreadsheet_name}_{timestamp}"

    sh_nova = gc.create(nome_spreadsheet)
    print(f"\nGoogle Spreadsheet nova criada com sucesso: {nome_spreadsheet}")

    if share_email is not None:
        sh_nova.share(share_email, perm_type="user", role="writer")
        print(f"Spreadsheet nova partilhada com: {share_email}")

    # 5) Abrir spreadsheet fixa
    sh_fixa = gc.open_by_key(spreadsheet_fixa_id)
    print(f"Spreadsheet fixa aberta com sucesso: {sh_fixa.title}")

    # 6) Exportar para spreadsheet nova
    print("\n=== EXPORTAÇÃO PARA SPREADSHEET NOVA ===")
    resumo_nova = exportar_para_uma_spreadsheet(
        conn,
        sh_nova,
        tabelas,
        apagar_sheet1=True
    )

    # 7) Atualizar spreadsheet fixa
    print("\n=== ATUALIZAÇÃO DA SPREADSHEET FIXA ===")
    resumo_fixa = exportar_para_uma_spreadsheet(
        conn,
        sh_fixa,
        tabelas,
        apagar_sheet1=False
    )

    # 8) Resumo final
    resumo_final = pd.concat([resumo_nova, resumo_fixa], ignore_index=True)

    print("\n=== RESUMO FINAL ===")
    print(resumo_final)

    print("\nSpreadsheet nova:", sh_nova.url)
    print("Spreadsheet fixa:", sh_fixa.url)

    clts.elapt["Google Spreadsheets export successful ✅"] = clts.deltat(tstart)

except Exception as e:
    print("Erro durante a execução:", e)
    clts.elapt[f"Execution error ❌: {e}"] = clts.deltat(tstart)

finally:
    if conn is not None:
        print("\nConnection closing....")
        conn.close()
        print("Ligação fechada.")

Ligação à base de dados com sucesso!
Autenticação Google com sucesso!

Tabelas existentes na base de dados com 'catalog' no nome:

1. catalog - 28 colunas
2. catalog_distribution - 18 colunas
3. catalog_internal - 10 colunas
4. catalog_internal_v2 - 10 colunas
5. catalog_keyword - 3 colunas
6. catalog_v2 - 28 colunas

Google Spreadsheet nova criada com sucesso: database_export_20260622_100620
Spreadsheet nova partilhada com: ana.carolinamb2001@gmail.com
Spreadsheet fixa aberta com sucesso: Confirmed

=== EXPORTAÇÃO PARA SPREADSHEET NOVA ===

A exportar tabela: catalog -> database_export_20260622_100620
   id       uri alias                                              title  \
0   1   meteo01  None  Estação meteorológica da Cobertura verde do Fo...   
1   2  qualar01  None         Estação de qualidade do ar (FORUM DA MAIA)   
2   3  qualar02  None     Estação de qualidade do ar (Rua do Património)   
3   4  qualar03  None  Estação de qualidade do ar (R. Eng. Duarte Pac...   
4   5   ru